# Exercício 25 — Sobrevivência de pacientes com leucemia (Feigl e Zelen, 1965)

**Objetivo.** Supondo que o tempo de sobrevivência $T$ (semanas) segue uma distribuição **gama**, propor um modelo para o tempo médio de sobrevivência em função de $\log(\text{WBC})$ e de AG (1 = positivo, 0 = negativo), ajustá-lo no **GAMLSS**, fazer a análise de diagnóstico (`plot`, `wp`, `term.plot`) e interpretar as estimativas.

Roteiro:
1. Dados e análise descritiva
2. Especificação do modelo
3. Ajuste no GAMLSS e seleção do preditor
4. Interpretação das estimativas
5. Diagnóstico: `plot`, `wp`, `term.plot`
6. Comentários finais (incluindo a simplificação para o modelo exponencial)

## 1. Dados

Se o arquivo `sobrev.txt` estiver na pasta de trabalho ele é lido; caso contrário, os dados são digitados a partir da tabela do enunciado (17 pacientes AG+ e 16 AG−). Ajuste os nomes das colunas do `read.table` se o seu arquivo usar outros nomes.

In [ ]:
library(gamlss)
options(repr.plot.width = 9, repr.plot.height = 6, digits = 4)

if (file.exists("sobrev.txt")) {
  d <- read.table("sobrev.txt", header = TRUE)
  names(d) <- tolower(names(d))   # espera colunas: tempo, wbc, ag
} else {
  d <- data.frame(
    wbc = c(2300, 750, 4300, 2600, 6000, 10500, 10000, 17000, 5400, 7000, 9400,
            32000, 35000, 100000, 100000, 52000, 100000,            # AG positivo
            4400, 3000, 4000, 1500, 9000, 5300, 10000, 19000, 27000, 28000,
            31000, 26000, 21000, 79000, 100000, 100000),            # AG negativo
    tempo = c(65, 156, 100, 134, 16, 108, 121, 4, 39, 143, 56, 26, 22, 1, 1, 5, 65,
              56, 65, 17, 7, 16, 22, 3, 4, 2, 3, 8, 4, 3, 30, 4, 43),
    ag = c(rep(1, 17), rep(0, 16))
  )
}
d$lwbc <- log(d$wbc)
d$agf  <- factor(d$ag, levels = c(0, 1), labels = c("AG-", "AG+"))
str(d)

: 

### Análise descritiva

In [ ]:
aggregate(cbind(tempo, lwbc) ~ agf, data = d,
          FUN = function(x) c(media = mean(x), dp = sd(x), n = length(x)))

# Para a gama, Var(T) = sigma^2 * mu^2  => o coeficiente de variação é constante.
with(d, tapply(tempo, agf, function(x) sd(x) / mean(x)))

In [ ]:
par(mfrow = c(1, 2))
plot(tempo ~ lwbc, data = d, pch = 19, col = ifelse(ag == 1, "steelblue", "tomato"),
     xlab = "log(WBC)", ylab = "Tempo (semanas)", main = "Tempo vs log(WBC)")
legend("topright", c("AG+", "AG-"), pch = 19, col = c("steelblue", "tomato"), bty = "n")
plot(log(tempo) ~ lwbc, data = d, pch = 19, col = ifelse(ag == 1, "steelblue", "tomato"),
     xlab = "log(WBC)", ylab = "log(Tempo)", main = "Escala log")
par(mfrow = c(1, 1))

**Leitura descritiva.** Os tempos são positivos e fortemente assimétricos (vários tempos de 1 a 5 semanas e alguns acima de 100). Pacientes AG+ sobrevivem, em média, bem mais que AG−, e em ambos os grupos o tempo tende a cair quando o WBC aumenta. Na escala log a relação com $\log(\text{WBC})$ parece aproximadamente linear e com inclinações parecidas nos dois grupos — o que sugere ligação log e efeitos aditivos (sem interação), hipótese testada adiante. O coeficiente de variação em cada grupo está perto de 1, compatível com a gama (CV constante) e, em particular, com a exponencial (CV = 1).

## 2. Especificação do modelo

Para $i = 1,\dots,33$:

$$T_i \sim \text{Gama}(\mu_i, \sigma), \qquad E(T_i) = \mu_i, \qquad \text{Var}(T_i) = \sigma^2 \mu_i^2,$$

$$\log \mu_i = \beta_0 + \beta_1 \log(\text{WBC}_i) + \beta_2\, \text{AG}_i .$$

* **Componente aleatório:** gama — resposta contínua, positiva, assimétrica, com variância proporcional a $\mu^2$. Na parametrização `GA` do GAMLSS, $\sigma$ é o coeficiente de variação; no GLM clássico o parâmetro de dispersão é $\phi = \sigma^2$ e o parâmetro de forma é $\nu = 1/\sigma^2$.
* **Função de ligação:** log, em vez da canônica (recíproca). A ligação log garante $\hat\mu_i > 0$ para qualquer valor dos coeficientes e dá interpretação multiplicativa direta via $e^{\beta}$.
* **Preditor:** $\log(\text{WBC})$ e AG; também avaliamos a interação $\log(\text{WBC})\times\text{AG}$ e um modelo com $\sigma$ dependendo de AG.

## 3. Ajuste no GAMLSS

In [ ]:
m1 <- gamlss(tempo ~ lwbc + ag, family = GA, data = d, trace = FALSE)
summary(m1)

### Seleção do preditor: interação e dispersão variável

In [ ]:
m2 <- gamlss(tempo ~ lwbc * ag, family = GA, data = d, trace = FALSE)   # com interação
m3 <- gamlss(tempo ~ lwbc + ag, sigma.formula = ~ ag,                    # sigma depende de AG
             family = GA, data = d, trace = FALSE)
LR.test(m1, m2)
GAIC(m1, m2, m3)

* **Interação:** o teste da razão de verossimilhanças dá LRT ≈ 1,71 com 1 g.l. (p ≈ 0,19), e o AIC praticamente não muda (301,0 vs 301,3). Não há evidência de que o efeito do WBC dependa de AG: ficamos com **retas paralelas na escala log**.
* **$\sigma$ dependente de AG:** o AIC piora (303,0). Mantemos $\sigma$ constante.

O modelo final é `m1`.

## 4. Interpretação das estimativas

$$\log \hat\mu = 5{,}819 \;-\; 0{,}305\,\log(\text{WBC}) \;+\; 1{,}018\,\text{AG}, \qquad \hat\sigma = e^{0{,}0253} \approx 1{,}03.$$

In [ ]:
b <- coef(m1, "mu")
ep <- sqrt(diag(vcov(m1)))[1:3]     # erros-padrão dos coeficientes de mu
ic <- cbind(li = b - 1.96 * ep, ls = b + 1.96 * ep)   # IC de Wald 95%
round(cbind(estimativa = b, ep, ic, exp_beta = exp(b), exp_li = exp(ic[, 1]), exp_ls = exp(ic[, 2])), 3)

sig <- fitted(m1, "sigma")[1]
c(sigma = sig, phi = sig^2, forma_nu = 1 / sig^2)

# Efeito do WBC em escalas mais intuitivas
c(WBC_x_e = exp(b["lwbc"]), WBC_x2 = 2^b["lwbc"], WBC_x10 = 10^b["lwbc"])

**Interpretação (com os demais fatores fixos):**

* **AG ($\hat\beta_2 = 1{,}018$; p ≈ 0,011).** $e^{1{,}018} \approx 2{,}77$ (IC 95%: 1,33 a 5,75): o tempo médio de sobrevivência de um paciente AG+ é cerca de **2,8 vezes** o de um paciente AG− com o mesmo WBC. A presença da característica morfológica é um fator prognóstico favorável.
* **log(WBC) ($\hat\beta_1 = -0{,}305$; p ≈ 0,024).** Como o WBC entra em log, o coeficiente é uma **elasticidade**: um aumento de 1% no WBC reduz o tempo médio em aproximadamente 0,3%. Em termos práticos: dobrar o WBC multiplica o tempo médio por $2^{-0{,}305} \approx 0{,}81$ (queda de ~19%); multiplicar o WBC por 10 multiplica o tempo médio por $10^{-0{,}305} \approx 0{,}50$, ou seja, **reduz a sobrevivência média pela metade**. O IC 95% para $e^{\beta_1}$ (0,57 a 0,95) exclui 1.
* **Intercepto ($\hat\beta_0 = 5{,}82$).** Corresponde a $\log\mu$ para AG− com $\log(\text{WBC}) = 0$ (WBC = 1), fora do alcance dos dados; não tem interpretação prática isolada. Exemplo útil: para AG− com WBC = 10 000, $\hat\mu = \exp(5{,}82 - 0{,}305\cdot 9{,}21) \approx 20$ semanas; para AG+ com o mesmo WBC, $\approx 55$ semanas.
* **Dispersão ($\hat\sigma \approx 1{,}03$).** O CV estimado do tempo é ≈ 1, isto é, o desvio-padrão é da mesma ordem da média — grande variabilidade individual. O parâmetro de forma $\nu = 1/\sigma^2 \approx 0{,}95$ está muito próximo de 1, o que indica que a distribuição **exponencial** (caso particular da gama com $\nu = 1$) seria suficiente (ver seção 6).

> Obs.: o `glm(..., family = Gamma(link = "log"))` fornece praticamente as mesmas estimativas de $\beta$; os erros-padrão diferem um pouco porque o GLM estima $\phi$ pelo método de Pearson ($\hat\phi \approx 1{,}09$), enquanto o GAMLSS estima $\sigma$ por máxima verossimilhança ($\hat\sigma^2 \approx 1{,}05$).

In [ ]:
g1 <- glm(tempo ~ lwbc + ag, family = Gamma(link = "log"), data = d)
summary(g1)$coefficients
summary(g1)$dispersion

### Curvas ajustadas

In [ ]:
grid <- seq(min(d$lwbc), max(d$lwbc), length = 100)
mu_neg <- predict(m1, newdata = data.frame(lwbc = grid, ag = 0), type = "response", data = d)
mu_pos <- predict(m1, newdata = data.frame(lwbc = grid, ag = 1), type = "response", data = d)
plot(tempo ~ lwbc, data = d, pch = 19, col = ifelse(ag == 1, "steelblue", "tomato"),
     xlab = "log(WBC)", ylab = "Tempo (semanas)", main = "Tempo médio ajustado")
lines(grid, mu_pos, col = "steelblue", lwd = 2)
lines(grid, mu_neg, col = "tomato", lwd = 2)
legend("topright", c("AG+", "AG-"), col = c("steelblue", "tomato"), lwd = 2, pch = 19, bty = "n")

## 5. Análise de diagnóstico

O GAMLSS usa os **resíduos quantílicos aleatorizados normalizados** (Dunn e Smyth, 1996): $r_i = \Phi^{-1}\{F(t_i;\hat\mu_i,\hat\sigma)\}$. Se o modelo estiver correto, $r_i \sim N(0,1)$ independentes, qualquer que seja a distribuição da resposta. Todos os gráficos abaixo se baseiam neles.

### 5.1 `plot(m1)`

In [ ]:
plot(m1)

O `plot` produz quatro painéis e um resumo numérico dos resíduos:

* **Resíduos vs valores ajustados** e **resíduos vs índice**: nuvem aleatória em torno de 0, sem tendência nem forma de funil, praticamente toda dentro de $\pm 2$. Indica que a ligação log e o preditor linear capturam bem a média e que a variância $\sigma^2\mu^2$ está adequada (não sobra heterocedasticidade). No gráfico por índice não se vê diferença sistemática entre o bloco AG+ (obs. 1–17) e AG− (18–33).
* **Densidade estimada dos resíduos**: aproximadamente simétrica e unimodal; com n = 33 a estimativa é naturalmente irregular.
* **QQ-plot normal**: pontos próximos da reta, sem caudas pesadas evidentes.
* **Resumo numérico**: média ≈ −0,01 e variância ≈ 1,02 (esperado: 0 e 1); assimetria ≈ 0,16 (esperado: 0); curtose ≈ 2,0, um pouco abaixo de 3, ou seja, caudas um pouco mais leves que a normal — o que é comum em amostras pequenas e não preocupa; o coeficiente de correlação de Filliben (≈ 0,986) confirma a boa aderência ao QQ-plot. A densidade estimada tem um leve "ombro" à direita, também atribuível ao tamanho da amostra. O maior resíduo em módulo é o da obs. 33 (AG−, WBC = 100 000, 43 semanas; $r \approx 2{,}16$), seguido pelas obs. 14–15 (AG+, WBC = 100 000, 1 semana; $r \approx -1{,}75$). São valores moderados para uma amostra de 33, não caracterizando outliers.

In [ ]:
r <- resid(m1)
shapiro.test(r)
head(sort(abs(r), decreasing = TRUE), 5)

O teste de Shapiro–Wilk (p ≈ 0,39) não rejeita a normalidade dos resíduos quantílicos.

### 5.2 Worm plot — `wp(m1)`

In [ ]:
wp(m1)

O *worm plot* (van Buuren e Fredriks, 2001) é um QQ-plot "destendenciado": no eixo vertical está a diferença entre o quantil empírico e o teórico ($N(0,1)$). As curvas tracejadas são bandas aproximadas de 95%.

* **Todos os 33 pontos dentro das bandas**, e longe delas → nenhuma evidência de inadequação da distribuição gama.
* A curva ajustada (cúbica) fica próxima de zero, com nível ≈ 0 (média ok) e sem inclinação (variância ok). Há uma leve ondulação — o ponto mais à esquerda um pouco acima de zero e o extremo direito ligeiramente abaixo —, que corresponde às caudas levemente curtas apontadas pela curtose ≈ 2; é pequena frente às bandas e não justifica trocar de distribuição.

Regra de leitura útil: deslocamento vertical do verme indica média mal ajustada; inclinação, variância mal especificada; forma de U, assimetria; forma de S, problema nas caudas. Nenhum desses padrões aparece.

Também podemos verificar a adequação **por faixas da covariável** (worm plots condicionais em intervalos de log(WBC)):

In [ ]:
wp(m1, xvar = d$lwbc, n.inter = 3)

O painel superior mostra as três faixas de log(WBC) (11 observações cada; ordem dos painéis: de baixo à esquerda para cima). Nas três faixas os pontos ficam dentro das bandas. Na faixa de WBC baixo (painel inferior esquerdo) o verme tem leve inclinação negativa e, na faixa alta (painel superior), leve inclinação positiva — sinais fracos de que a dispersão relativa varia um pouco com o WBC —, mas com apenas 11 pontos por faixa nada disso é significativo. O modelo com $\sigma$ dependendo de log(WBC) poderia ser testado da mesma forma que `m3`.

### 5.3 `term.plot(m1)`

In [ ]:
term.plot(m1, what = "mu", partial.resid = TRUE, ask = FALSE, pages = 1)

O `term.plot` mostra a **contribuição de cada termo ao preditor linear** $\log\mu$ (centrada), com banda de ±2 erros-padrão e, aqui, os **resíduos parciais** sobrepostos.

* **log(WBC):** reta decrescente com inclinação $-0{,}305$; a banda de ±2 EP tem forma de "gravata-borboleta", mais larga nos extremos de log(WBC), onde há menos informação. Os resíduos parciais acompanham a tendência, mas há um padrão a notar: entre log(WBC) ≈ 9,7 e 10,5 (WBC entre ~16 000 e ~35 000) quase todos ficam **abaixo** da reta, e no extremo log(WBC) ≈ 11,5 (WBC = 100 000) eles se espalham muito, de −1,6 a +2,7. Esses pontos extremos são justamente os pacientes com WBC = 100 000, que provavelmente é um **valor de teto** de registro (WBC ≥ 100 000), não uma medida exata.
* **AG:** um degrau de 1,018 entre AG− e AG+ (o efeito centrado mostra os dois níveis); as bandas dos dois níveis não se sobrepõem, coerente com p ≈ 0,011. O resíduo parcial mais alto em AG− (≈ 2,7) é a obs. 33.

O padrão no painel de log(WBC) sugere verificar a linearidade com um suavizador P-spline penalizado:

In [ ]:
m4 <- gamlss(tempo ~ pb(lwbc) + ag, family = GA, data = d, trace = FALSE)
GAIC(m1, m4)
edf(m4)
m4b <- gamlss(tempo ~ pb(lwbc, method = "GAIC", k = log(33)) + ag, family = GA, data = d, trace = FALSE)
edf(m4b)
mq <- gamlss(tempo ~ poly(lwbc, 2) + ag, family = GA, data = d, trace = FALSE)
LR.test(m1, mq)
term.plot(m4, what = "mu", partial.resid = TRUE, ask = FALSE, pages = 1)

Com a seleção do suavizador por AIC (padrão do `pb`), a curva usa ≈ 4,4 g.l. efetivos e o AIC cai de 301,0 para 296,8: há **algum sinal de não linearidade**, com a curva caindo mais forte entre WBC ≈ 10 000 e 30 000 e voltando a subir no extremo de WBC = 100 000. Porém:

* esse "repique" no final é sustentado apenas pelos pacientes com WBC no teto de 100 000, cujo valor real é desconhecido;
* um termo quadrático não é significativo (LRT ≈ 1,48; p ≈ 0,22);
* se o grau de suavização for escolhido pelo critério bayesiano, `pb(lwbc, method = "GAIC", k = log(33))`, os g.l. efetivos caem para 2 — no `pb` isso corresponde a uma reta (intercepto e inclinação) —, recuperando exatamente o modelo linear (mesmo AIC de `m1`).

Com n = 33, mantemos o efeito **linear** em log(WBC), que é o modelo pedido no enunciado e tem interpretação simples, registrando a curvatura como ponto a investigar com mais dados.

## 6. Comentários finais

### 6.1 Gama ou exponencial?

Como $\hat\sigma \approx 1$, testamos $H_0:\sigma = 1$ (exponencial) contra a gama:

In [ ]:
m_exp <- gamlss(tempo ~ lwbc + ag, family = EXP, data = d, trace = FALSE)
LR.test(m_exp, m1)
GAIC(m_exp, m1)
coef(m_exp)

LRT ≈ 0,06 (p ≈ 0,81) e AIC menor para a exponencial (299,1 vs 301,0): a restrição $\sigma = 1$ é plenamente compatível com os dados. Os coeficientes de $\mu$ são idênticos (a estimativa de $\beta$ na gama não depende de $\sigma$), então **as interpretações da seção 4 continuam válidas**; ganha-se apenas parcimônia. Esse é, aliás, o modelo usado originalmente por Feigl e Zelen (1965).

### 6.2 Sensibilidade à observação mais extrema

In [ ]:
m1_s33 <- gamlss(tempo ~ lwbc + ag, family = GA, data = d[-33, ], trace = FALSE)
round(rbind(completo = coef(m1), sem_obs33 = coef(m1_s33)), 3)

Sem a obs. 33 o efeito do WBC fica **mais forte** ($-0{,}42$ em vez de $-0{,}31$) e o de AG também ($1{,}23$ em vez de $1{,}02$). Esse paciente AG− com WBC muito alto que sobreviveu 43 semanas "puxa" as estimativas para zero; os sinais e as conclusões não mudam, então ele não é problemático, apenas influente — e removê-lo não se justifica, pois não há indício de erro de registro.

### 6.3 Conclusão

* O modelo gama com ligação log, $\log\mu = \beta_0 + \beta_1\log(\text{WBC}) + \beta_2\text{AG}$, está bem ajustado: os resíduos quantílicos se comportam como $N(0,1)$ e o worm plot fica inteiramente dentro das bandas. O `term.plot` mostra uma possível curvatura em log(WBC), fraca e dependente dos pacientes com WBC no teto de 100 000; a forma linear foi mantida.
* **AG+ multiplica o tempo médio de sobrevivência por ≈ 2,8**, para o mesmo WBC.
* **Cada aumento de 10 vezes no WBC reduz o tempo médio pela metade** (dobrar o WBC reduz ≈ 19%), em ambos os grupos.
* A variabilidade é grande (CV ≈ 1) e a distribuição exponencial é uma simplificação adequada da gama para estes dados.
* Limitação: n = 33 e não há censura nos dados; em estudos atuais de sobrevivência, censuras exigiriam modelos próprios (p.ex. `gamlss.cens` ou regressão de Cox/Weibull).